
- Walkthrough clustering sample::

https://www.databricks.com/notebooks/segment-p13n/sg_03_clustering.html

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_gold.tbg_customer_item_clustering limit 5

In [0]:
%sql
SELECT DISTINCT order_id, zipcode, latitude, longitude, type, value FROM end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders limit 3

In [0]:
df = spark.sql("""
SELECT DISTINCT order_id, zipcode, latitude, longitude, type, value FROM end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders
""")

#### 1) Data sanitization ::

In [0]:
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.model_selection import train_test_split
from scipy.cluster.hierarchy import dendrogram, set_link_color_palette
 
import numpy as np
import pandas as pd
 
import mlflow
import os
 
from delta.tables import *
 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors
import seaborn as sns

In [0]:
from pyspark.ml.feature import StringIndexer

# StringIndexer to convert 'type' to numeric indices
indexer = StringIndexer(inputCol="type", outputCol="type_index")
df_indexed = indexer.fit(df).transform(df)

# Get distinct type values for manual one-hot encoding
type_values = [row['type'] for row in df.select('type').distinct().collect()]

# Create one-hot columns for each type
for t in type_values:
    col_name = f"type_{t}"
    df_indexed = df_indexed.withColumn(col_name, (df_indexed['type'] == t).cast("int"))

display(df_indexed)

In [0]:
df_pd_saved = df_indexed.toPandas()
df_pd_saved = df_pd_saved.reset_index()
display(df_pd_saved)

In [0]:
df_indexed_no_order_id = df_pd_saved.drop(['order_id', 'type'], axis=1)

####  2) Isolation Forest for outliers treatment

In [0]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

# Convert Spark DataFrame to Pandas for sklearn processing
df_pd = df_indexed_no_order_id

# 2. Defensive Sanitization: Identify and remove anomalies
iso_forest = IsolationForest(contamination=0.05, random_state=43)
outlier_preds = iso_forest.fit_predict(df_pd)

# Filter: keep only "inliers" (1), discard outliers (-1)
sanitized_data = df_pd[outlier_preds == 1]

# 3. Perform Clustering on Sanitized Data
kmeans = KMeans(n_clusters=3, n_init=10)
clusters = kmeans.fit_predict(sanitized_data)

print(f"Original points: {len(df_pd)}")
print(f"Sanitized points: {len(sanitized_data)}")
print(f"Final Cluster Centers:\n{kmeans.cluster_centers_}")


#### 3) After treating outliers, scaling the dataset good idea

In [0]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

df = pd.DataFrame(df_pd)

# 2. Handle Missing Values (Imputation)
# Replacing NaNs with the mean to maintain data originality
imputer = SimpleImputer(strategy='mean')
clean_data = imputer.fit_transform(df)

# 3. Scaling & Normalization
# Ensuring 'on variable' doesn't dominate the other due to magnitude
scaler = MinMaxScaler()
sanitized_scaled_data = scaler.fit_transform(clean_data)

display(sanitized_scaled_data)

In [0]:
sanitized_scaled_data = pd.DataFrame(sanitized_scaled_data, columns=df.columns)

In [0]:
type(sanitized_scaled_data)

In [0]:
# initial cluster count
initial_n = 5
 
# train the model
initial_model = KMeans(
  n_clusters=initial_n,
  max_iter=1000
  )
 
# fit and predict per-household cluster assignment
init_clusters = initial_model.fit_predict(sanitized_scaled_data)
 
# combine households with cluster assignments
labeled_sanitized_scaled_data_pd = (
  pd.concat( 
    [sanitized_scaled_data, pd.DataFrame(init_clusters,columns=['cluster'])],
    axis=1
    )
  )
 
# visualize cluster assignments
fig, ax = plt.subplots(figsize=(5,4))
sns.scatterplot(
  data=labeled_sanitized_scaled_data_pd,
  x='latitude',
  y='longitude',
  hue='cluster',
  palette=[cm.nipy_spectral(float(i) / initial_n) for i in range(initial_n)],
  legend='brief',
  alpha=0.5,
  ax = ax
  )
_ = ax.legend(loc='lower right', ncol=1, fancybox=True)

In [0]:
# function to train model and return metrics
def evaluate_model(n):
    model = KMeans(n_clusters=n, init='k-means++', n_init=1, max_iter=1000)
    clusters = model.fit(sanitized_scaled_data).labels_
    return n, float(model.inertia_), float(silhouette_score(sanitized_scaled_data, clusters))

# evaluate models for each value of k
results = [evaluate_model(n) for n in range(2, 21)]

results_pd = pd.DataFrame(results, columns=['n', 'inertia', 'silhouette'])

display(results_pd)

In [0]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6,4))
ax.plot(results_pd['n'], results_pd['inertia'], marker='o')
ax.set_xlabel('Cluster Count (n)')
ax.set_ylabel('Inertia')
ax.set_title('Inertia over Cluster Count')
plt.tight_layout()
# display(fig)

In [0]:
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(results_pd['n'], results_pd['silhouette'], marker='o', color='orange')
ax.set_xlabel('Cluster Count (n)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score over Cluster Count')
plt.tight_layout()
# display(fig)

In [0]:
total_iterations = 500
n_for_bestofk = 7

best_score = -1
best_model = None

for i in range(total_iterations):
    model = KMeans(n_clusters=n_for_bestofk, n_init=1, init='k-means++', max_iter=100)
    model.fit(sanitized_scaled_data)
    score = float(silhouette_score(sanitized_scaled_data, model.labels_))
    if score > best_score:
        best_score = score
        best_model = model

bestofk_score = best_score
bestofk_model = best_model
bestofk_clusters = bestofk_model.labels_

print('Silhouette Score: {0:.6f}'.format(bestofk_score))

bestofk_labeled_X_pd = (
    pd.concat(
        [sanitized_scaled_data, pd.DataFrame(bestofk_clusters, columns=['cluster'])],
        axis=1
    )
)

### 4) Visualize Best of K Clusters

In [0]:
# visualize cluster assignments
fig, ax = plt.subplots(figsize=(5,4))
sns.scatterplot(
  data=bestofk_labeled_X_pd,
  x='latitude',
  y='longitude',
  hue='cluster',
  palette=[cm.nipy_spectral(float(i) / n_for_bestofk) for i in range(n_for_bestofk)],  # align colors with those used in silhouette plots
  legend='brief',
  alpha=0.5,
  ax = ax
  )
_ = ax.legend(loc='lower right', ncol=1, fancybox=True)

In [0]:
display(df_pd.head(3))

In [0]:
from sklearn.metrics import silhouette_samples

# Compute per-member silhouette scores
member_silhouette_scores = silhouette_samples(sanitized_scaled_data, bestofk_clusters)
avg_score = member_silhouette_scores.mean()

# Add scores to DataFrame
bestofk_labeled_X_pd['silhouette_score'] = member_silhouette_scores

# Silhouette plot
fig, ax = plt.subplots(figsize=(7, 5))
y_lower = 10
for i in range(n_for_bestofk):
    ith_cluster_silhouette_values = member_silhouette_scores[bestofk_clusters == i]
    ith_cluster_silhouette_values.sort()
    size_cluster = ith_cluster_silhouette_values.shape[0]
    y_upper = y_lower + size_cluster
    color = cm.nipy_spectral(float(i) / n_for_bestofk)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_values, facecolor=color, edgecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_cluster, str(i))
    y_lower = y_upper + 10

ax.axvline(avg_score, color="red", linestyle="--", label=f"Average: {avg_score:.3f}")
ax.set_xlabel("Silhouette Score")
ax.set_ylabel("Cluster Label")
ax.set_title("Silhouette Plot for Each Cluster")
ax.legend()
plt.tight_layout()

- Better to use 2 clusters.

In [0]:
bestofk_labeled_X_pd.head(3)

In [0]:
# Inverse MinMaxScaler effect
scaler = MinMaxScaler()
scaler.fit(df_indexed_no_order_id)
unscaled_data = pd.DataFrame(
    scaler.inverse_transform(bestofk_labeled_X_pd[df_indexed_no_order_id.columns]),
    columns=df_indexed_no_order_id.columns
)

# Round up index column
if 'index' in unscaled_data.columns:
    unscaled_data['index'] = np.ceil(unscaled_data['index']).astype(int)

# Left join using index, remove duplicate columns
merged = unscaled_data.join(
    bestofk_labeled_X_pd.drop(columns=df_indexed_no_order_id.columns),
    how='left'
)

display(merged)

In [0]:
df_pd_saved.head(3)

In [0]:
merged = merged.merge(df_pd_saved, left_on='index', right_on='index', how='left', suffixes=('', '_dup'))
# Remove duplicate columns (those ending with '_dup')
merged = merged.loc[:, ~merged.columns.str.endswith('_dup')]
display(merged)

In [0]:
spark.createDataFrame(merged).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders_clustered')

#### Sillhouete score
##### Interpretation of Values, the score ranges from -1 to +1:

- Close to +1: Indicates the point is far away from neighboring clusters and well-matched to its own.
- Rule of thumb: A score \(>0.5\) suggests a strong or "reasonable" structure.
- Near 0: Indicates the point is on or very close to the decision boundary between two clusters (overlapping clusters).
- Close to -1: Indicates the point was likely assigned to the wrong cluster, as it is more similar to a neighboring group

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders_clustered limit 3

### 5) Hierarchical Clustering

source ::

- https://www.databricks.com/notebooks/segment-p13n/sg_03_clustering.html
- https://scikit-learn.org/stable/auto_examples/cluster/plot_agglomerative_dendrogram.html#sphx-glr-auto-examples-cluster-plot-agglomerative-dendrogram-py

In [0]:
# function to generate dendrogram
def plot_dendrogram(model, **kwargs):
 
    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count
 
    linkage_matrix = np.column_stack(
                      [model.children_, 
                       model.distances_,
                       counts]
                      ).astype(float)
 
    # Plot the corresponding dendrogram
    j = 5
    set_link_color_palette(
      [matplotlib.colors.rgb2hex(cm.nipy_spectral(float(i) / j)) for i in range(j)]
      )
    dendrogram(linkage_matrix, **kwargs)

#### The four primary linkage types are:


- **Ward Linkage (Default)**: Minimizes the sum of squared differences within all clusters. It merges the two clusters that result in the smallest increase in total within-cluster variance. This method typically produces compact, relatively equal-sized, and spherical clusters.

- **Complete Linkage (Maximum Linkage)**: Defines the distance between two clusters as the maximum distance between any single data point in the first cluster and any single data point in the second. It seeks to merge clusters to keep the maximum diameter of the resulting cluster as small as possible, often leading to compact, spherical shapes.

- **Average Linkage**: Calculates the distance between two clusters as the average distance of each observation from one set to every observation in the other set. It serves as a balance between Single and Complete linkage and is often used to create clusters of similar sizes.

- **Single Linkage (Minimum Linkage)**: Defines the distance between two clusters as the minimum distance (closest points) between any pair of observations from the two clusters. While it can handle non-globular shapes, it is prone to the "chaining effect," where thin bridges of noise can cause two distinct clusters to merge into one long, irregular cluster.

In [0]:
df_indexed_no_order_id.head(3)

In [0]:
# train cluster model
inithc_model = AgglomerativeClustering(distance_threshold=0, n_clusters=None, linkage='ward')
inithc_model.fit(df_indexed_no_order_id)
 
# generate visualization
fig, ax = plt.subplots(1, 1)
fig.set_size_inches(15, 8)
 
plot_dendrogram(inithc_model, truncate_mode='level', p=6) # 6 levels max
plt.title('Hierarchical Clustering Dendrogram')
_ = plt.xlabel('Number of points in node (or index of point if no parenthesis)')

In [0]:
# Identify Number of Clusters
results = []
 
# train models with n number of clusters * linkages
for a in ['ward']:  # linkages ['ward', 'complete', 'average', 'single' ]
  for n in range(2,15): # evaluate 2 to 14 clusters (minimum 2 required for silhouette)
 
    # fit the algorithm with n clusters
    model = AgglomerativeClustering(n_clusters=n, linkage=a)
    clusters = model.fit(df_indexed_no_order_id).labels_
 
    # capture the inertia & silhouette scores for this value of n
    results += [ (n, a, silhouette_score(df_indexed_no_order_id, clusters)) ]
 
results_pd = pd.DataFrame(results, columns=['n', 'linkage', 'silhouette'])
display(results_pd)

In [0]:
plt.figure(figsize=(8, 5))
plt.plot(results_pd['n'], results_pd['silhouette'], marker='o')
plt.xlabel('Number of Clusters (n)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs. Number of Clusters')
plt.grid(True)
plt.tight_layout()

#### Train & Evaluate Model

In [0]:
n_for_besthc = 13
linkage_for_besthc = 'ward'
 
# configure model
besthc_model = AgglomerativeClustering(n_clusters=n_for_besthc, linkage=linkage_for_besthc)
 
# train and predict clusters
besthc_clusters = besthc_model.fit(df_indexed_no_order_id).labels_
 
# score results
besthc_score = silhouette_score(df_indexed_no_order_id, besthc_clusters)
 
# print best score obtained
print('Silhouette Score: {0:.6f}'.format(besthc_score))
 
# combine customer with cluster assignments
besthc_labeled_X_pd = (
  pd.concat( 
    [df_indexed_no_order_id, pd.DataFrame(besthc_clusters,columns=['cluster'])],
    axis=1
    )
  )

In [0]:
besthc_labeled_X_pd

In [0]:
from sklearn.metrics import silhouette_samples

# Compute per-member silhouette scores
member_silhouette_scores = silhouette_samples(besthc_labeled_X_pd, bestofk_clusters)
avg_score = member_silhouette_scores.mean()

# Add scores to DataFrame
bestofk_labeled_X_pd['silhouette_score'] = member_silhouette_scores

# Silhouette plot
fig, ax = plt.subplots(figsize=(7, 5))
y_lower = 10
for i in range(n_for_bestofk):
    ith_cluster_silhouette_values = member_silhouette_scores[bestofk_clusters == i]
    ith_cluster_silhouette_values.sort()
    size_cluster = ith_cluster_silhouette_values.shape[0]
    y_upper = y_lower + size_cluster

    # Selecting the best "n_for_bestofk" silhouette score
    color = cm.nipy_spectral(float(i) / n_for_bestofk)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_values, facecolor=color, edgecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_cluster, str(i))
    y_lower = y_upper + 10

ax.axvline(avg_score, color="red", linestyle="--", label=f"Average: {avg_score:.3f}")
ax.set_xlabel("Silhouette Score")
ax.set_ylabel("Cluster Label")
ax.set_title("Silhouette Plot for Each Cluster")
ax.legend()
plt.tight_layout()

In [0]:
# visualize cluster assignments
fig, ax = plt.subplots(figsize=(5,4))
sns.scatterplot(
  data=bestofk_labeled_X_pd,
  x='latitude',
  y='longitude',
  hue='cluster',
  palette=[cm.nipy_spectral(float(i) / n_for_bestofk) for i in range(n_for_bestofk)],  # align colors with those used in silhouette plots
  legend='brief',
  alpha=0.5,
  ax = ax
  )
_ = ax.legend(loc='lower right', ncol=1, fancybox=True)

In [0]:
spark.createDataFrame(besthc_labeled_X_pd).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders_hc_cluster')

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders_hc_cluster limit 5

In [0]:
%sql
SELECT cluster, count(cluster) FROM end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders_hc_cluster
GROUP BY cluster
HAVING sum(cluster)

- The cluster can trigger an agent to execute a task, ai_classify().

In [0]:
%sql
SELECT cluster, count(cluster) FROM end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders_clustered 
GROUP BY cluster
HAVING sum(cluster)

In [0]:
%sql
SELECT cluster, count(cluster) FROM end_to_end_retail.db_gold.tbg_customer_item_clustering_sales_orders_hc_cluster
GROUP BY cluster
HAVING sum(cluster)